# Batch Synthesize

Run batch DDSP timbre transfer and vocoder baselines on the full voice dataset.

1. **DDSP inference** — `synthesize_dir` processes all files in `data/raw/voice/FULL/` through the trained DDSP model.
2. **WORLD baseline** — `vocode_dir` processes the same dataset using the WORLD vocoder with a source bank from `data/raw/solo_violin/`.
3. **SMS/HPS baseline** — `vocode_dir_sms` uses the Harmonic Plus Stochastic model (`sms-tools`) with the same source bank.

In [1]:
import logging
import sys
from pathlib import Path


In [2]:

PROJECT_ROOT = Path.cwd().parent.parent.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")

print(f"Project root: {PROJECT_ROOT}")

Project root: /m/home/home3/37/thieun1/unix/Project/final_project


In [3]:
from evaluation.batch_inference import synthesize_dir, vocode_dir, vocode_dir_sms

2026-04-16 13:43:52.579302: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-16 13:43:52.674301: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-16 13:44:01.509976: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /u/37/thieun1/unix/anaconda3/envs/conda_env3.10/lib
2026-04-16 13:44:01.510151: W tensorflow/compi

## Pause / Resume

The pipeline automatically resumes from where it left off — already-processed files are skipped on re-run.

- **To pause**: interrupt the kernel (`Kernel > Interrupt` or `I, I` in Jupyter). Files written so far are preserved.
- **To resume**: just re-run the pipeline cell. Completed files are detected and skipped automatically.
- **`skipped`** in the output = files whose output already existed (valid WAV > 44 bytes).

## Paths

Adjust these if your directory layout differs.

In [4]:
# --- Input ---
INPUT_DIR = PROJECT_ROOT / "data" / "raw" / "voice" / "FULL"
SOURCE_DIR = PROJECT_ROOT / "data" / "raw" / "solo_violin"

# --- DDSP model ---
MODEL_DIR = PROJECT_ROOT / "artifacts" / "solo_instrument_noreverb"
GIN_FILE = MODEL_DIR / "operative_config-0.gin"

# --- Output ---
DDSP_OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "voice" / "Full_transfered_noreverb"
BASELINE_OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "voice" / "Full_baseline"
SMS_OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "voice" / "Full_sms_baseline"

print(f"Input dir:    {INPUT_DIR}  (exists: {INPUT_DIR.exists()})")
print(f"Source dir:   {SOURCE_DIR}  (exists: {SOURCE_DIR.exists()})")
print(f"Model dir:    {MODEL_DIR}  (exists: {MODEL_DIR.exists()})")
print(f"Gin file:     {GIN_FILE}  (exists: {GIN_FILE.exists()})")
print(f"DDSP output:  {DDSP_OUTPUT_DIR}")
print(f"BL output:    {BASELINE_OUTPUT_DIR}")
print(f"SMS output:   {SMS_OUTPUT_DIR}")

Input dir:    /m/home/home3/37/thieun1/unix/Project/final_project/data/raw/voice/FULL  (exists: True)
Source dir:   /m/home/home3/37/thieun1/unix/Project/final_project/data/raw/solo_violin  (exists: True)
Model dir:    /m/home/home3/37/thieun1/unix/Project/final_project/artifacts/solo_instrument_noreverb  (exists: True)
Gin file:     /m/home/home3/37/thieun1/unix/Project/final_project/artifacts/solo_instrument_noreverb/operative_config-0.gin  (exists: True)
DDSP output:  /m/home/home3/37/thieun1/unix/Project/final_project/data/processed/voice/Full_transfered_noreverb
BL output:    /m/home/home3/37/thieun1/unix/Project/final_project/data/processed/voice/Full_baseline
SMS output:   /m/home/home3/37/thieun1/unix/Project/final_project/data/processed/voice/Full_sms_baseline


## 1. DDSP Timbre Transfer

Loads the model once, then processes all WAV files via feature-level chunking.

In [5]:
# ddsp_result = synthesize_dir(
#     model_dir=MODEL_DIR,
#     gin_file=GIN_FILE,
#     input_dir=INPUT_DIR,
#     output_dir=DDSP_OUTPUT_DIR,
#     auto_adjust=False,
#     pitch_shift=0.0,
#     loudness_shift=0.0,
# )

# print(f"\nDDSP — processed: {ddsp_result['processed']}, skipped: {ddsp_result['skipped']}, failed: {ddsp_result['failed']}")
# if ddsp_result["failed_files"]:
#     print("Failed files:")
#     for f in ddsp_result["failed_files"]:
#         print(f"  {f}")

## 2. WORLD Vocoder Baseline

Builds a source bank from solo violin recordings, then runs F0 transfer on all target files.

In [6]:
import os

# Set n_workers > 1 to parallelise across CPU cores.
# 1 = sequential (default, safest). os.cpu_count() = use all cores.
N_WORKERS = max(1, (os.cpu_count() or 1) // 2)

# Use CREPE for target F0 estimation instead of WORLD's Harvest+Stonemask.
# More robust on voice, but slower and requires TensorFlow.
# USE_CREPE = True

# baseline_result = vocode_dir(
#     input_dir=INPUT_DIR,
#     output_dir=BASELINE_OUTPUT_DIR,
#     source_dir=SOURCE_DIR,
#     method="f0_ap",
#     seed=42,
#     n_workers=N_WORKERS,
#     use_crepe=USE_CREPE,
# )

# print(f"\nBaseline — processed: {baseline_result['processed']}, skipped: {baseline_result['skipped']}, failed: {baseline_result['failed']}")
# print(f"  (ran with {N_WORKERS} workers, use_crepe={USE_CREPE})")
# if baseline_result["failed_files"]:
#     print("Failed files:")
#     for f in baseline_result["failed_files"]:
#         print(f"  {f}")

## 3. SMS/HPS Baseline

Uses the Harmonic Plus Stochastic model from `sms-tools` for timbre transfer.
Same two-phase approach: CREPE F0 precomputed on the main process (GPU), then
SMS analysis/synthesis distributed to CPU workers.

In [7]:
import os

N_WORKERS_SMS = max(1, (os.cpu_count() or 1) // 2)
USE_CREPE_SMS = True

sms_result = vocode_dir_sms(
    input_dir=INPUT_DIR,
    output_dir=SMS_OUTPUT_DIR,
    source_dir=SOURCE_DIR,
    method="f0_ap",
    alpha=1.0,
    seed=42,
    n_workers=N_WORKERS_SMS,
    use_crepe=USE_CREPE_SMS,
    # SMS/HPS analysis parameters (defaults are fine for most cases)
    # window="blackmanharris",
    # M=1001, N=2048, H=256, Ns=512,
    # n_harm=20, min_f0=80.0, max_f0=1200.0,
    # t=-80, f0et=5.0, stocf=0.1,
)

print(f"\nSMS — processed: {sms_result['processed']}, skipped: {sms_result['skipped']}, failed: {sms_result['failed']}")
print(f"  (ran with {N_WORKERS_SMS} workers, use_crepe={USE_CREPE_SMS})")
if sms_result["failed_files"]:
    print("Failed files:")
    for f in sms_result["failed_files"]:
        print(f"  {f}")

2026-04-16 13:44:43,236 INFO evaluation.batch_inference: SMS pipeline: 3095 files in /m/home/home3/37/thieun1/unix/Project/final_project/data/raw/voice/FULL
2026-04-16 13:44:51,473 INFO evaluation.batch_inference: Source bank: II. Double.wav → 2994176 samples
2026-04-16 13:44:51,753 INFO evaluation.batch_inference: Source bank: III. Corrente.wav → 3212544 samples
2026-04-16 13:44:51,940 INFO evaluation.batch_inference: Source bank: IV. Double Presto.wav → 2222848 samples
2026-04-16 13:44:52,114 INFO evaluation.batch_inference: Source bank: V. Sarabande.wav → 2009344 samples
2026-04-16 13:44:52,301 INFO evaluation.batch_inference: Source bank: VIII. Double.wav → 2183424 samples
2026-04-16 13:44:52,376 INFO evaluation.batch_inference: Precomputing CREPE F0 for 3095 files (sequential, main process)
CREPE F0 precompute:   0%|          | 0/3095 [00:00<?, ?it/s]2026-04-16 13:44:53.165849: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:981] successful NUMA node read from 


SMS — processed: 3095, skipped: 0, failed: 0
  (ran with 8 workers, use_crepe=True)


## Results Summary

In [9]:
print("=" * 50)
print("Batch Synthesis Results")
print("=" * 50)
# print(f"DDSP:     {ddsp_result['processed']} new / {ddsp_result['skipped']} skipped / {ddsp_result['failed']} failed")
# print(f"Baseline: {baseline_result['processed']} new / {baseline_result['skipped']} skipped / {baseline_result['failed']} failed")
print(f"SMS:      {sms_result['processed']} new / {sms_result['skipped']} skipped / {sms_result['failed']} failed")
print(f"\nOutputs saved to:")
# print(f"  DDSP:     {DDSP_OUTPUT_DIR}")
print(f"  Baseline: {BASELINE_OUTPUT_DIR}")
print(f"  SMS:      {SMS_OUTPUT_DIR}")

Batch Synthesis Results
SMS:      3095 new / 0 skipped / 0 failed

Outputs saved to:
  Baseline: /m/home/home3/37/thieun1/unix/Project/final_project/data/processed/voice/Full_baseline
  SMS:      /m/home/home3/37/thieun1/unix/Project/final_project/data/processed/voice/Full_sms_baseline
